In [1]:
!pip install selenium pandas webdriver-manager beautifulsoup4

envs/py1/lib/python3.9/site-packages/traitlets/traitlets.py", line 722, in _validate
    value = self.validate(obj, value)
  File "/opt/anaconda3/envs/py1/lib/python3.9/site-packages/traitlets/traitlets.py", line 2311, in validate
    self.error(obj, value)
  File "/opt/anaconda3/envs/py1/lib/python3.9/site-packages/traitlets/traitlets.py", line 831, in error
    raise TraitError(e)
traitlets.traitlets.TraitError: The '_control_lock' trait of an IPythonKernel instance expected a Lock, not the NoneType None.


In [5]:
import sys
!{sys.executable} -m pip install selenium pandas webdriver-manager beautifulsoup4 lxml

  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 5.0 MB/s  0:00:01 eta 0:00:01
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [39]:
import time
import pandas as pd
from io import StringIO
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# This script scrapes 13 months of data to allow for Month-over-Month calculation

def scrape_cpi_13months():
    # Target URL: Statistics Canada CPI Table 18-10-0004-08
    url = "https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810000408"
    
    options = Options()
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    final_data_list = []

    try:
        driver.get(url)
        wait = WebDriverWait(driver, 30)
        print("🔄 Initializing Final Scraper (13 Months / Vertical Stacking / Year-Overflow)...")

        # 1. Expand 'Customize table' to access date dropdowns
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "details")))
        driver.execute_script("document.querySelector('details').setAttribute('open', 'open')")
        time.sleep(2)
        
        # 2. Detect the current/default Month and Year indices
        month_el = driver.find_element(By.NAME, "cubeTimeFrame.startMonth")
        year_el = driver.find_element(By.NAME, "cubeTimeFrame.startYear")
        
        initial_mo_idx = int(month_el.get_attribute("selectedIndex"))
        initial_yr_idx = int(year_el.get_attribute("selectedIndex"))

        # 3. Main Loop: Move backward for exactly 13 months
        for i in range(13):
            target_mo_idx = initial_mo_idx - i
            target_yr_idx = initial_yr_idx
            
            # Logic for Year/Month Auto-Adjustment
            # When month goes below January (Index 0), move the Year index back by 1 (2025 -> 2024)
            # and reset month index to 11 (December)
            while target_mo_idx < 0:
                target_mo_idx += 12
                target_yr_idx -= 1  # Move to the previous year in your HTML list

            # Re-locate elements to prevent StaleElementReferenceException
            driver.execute_script("document.querySelector('details').setAttribute('open', 'open')")
            month_el = driver.find_element(By.NAME, "cubeTimeFrame.startMonth")
            year_el = driver.find_element(By.NAME, "cubeTimeFrame.startYear")
            
            # Select Year first to refresh the month list
            Select(year_el).select_by_index(target_yr_idx)
            driver.execute_script("arguments[0].dispatchEvent(new Event('change', { bubbles: true }));", year_el)
            time.sleep(1.5) # Wait for the site to process the year change
            
            # Select the target Month
            month_el = driver.find_element(By.NAME, "cubeTimeFrame.startMonth")
            Select(month_el).select_by_index(target_mo_idx)
            driver.execute_script("arguments[0].dispatchEvent(new Event('change', { bubbles: true }));", month_el)
            
            # Identify month/year name for the logs
            mo_name = Select(month_el).options[target_mo_idx].text
            yr_name = Select(year_el).options[target_yr_idx].text
            
            # Click 'Apply' to refresh the data table
            driver.find_element(By.ID, "applyButton").click()
            print(f"🔄 [{i+1}/13] Fetching: {mo_name} {yr_name}...")
            
            # Wait for data table to load numerical values
            time.sleep(11)

            # 4. Extract Data from the 4th Column (iloc[3])
            html_content = driver.page_source
            df_list = pd.read_html(StringIO(html_content))
            
            for df in df_list:
                if "Products and product groups" in str(df.columns):
                    # Target pharmaceutical categories
                    targets = ["Prescribed medicines", "Non-prescribed medicines"]
                    filtered = df[df.iloc[:, 0].str.contains('|'.join(targets), na=False, case=False)].copy()
                    
                    for _, row in filtered.iterrows():
                        # Extract row-by-row for vertical stacking
                        final_data_list.append({
                            "Category": row.iloc[0],
                            "Value": row.iloc[3], # Target numerical value column
                            "Month": mo_name,
                            "Year": yr_name
                        })
                    break

        # 5. Save final result to CSV
        if final_data_list:
            result_df = pd.DataFrame(final_data_list)
            result_df.to_csv("CPI_last13months.csv", index=False, encoding="utf-8-sig")
            print("\n✨ Done! 13-month report saved as CPI_last13months.csv")
        else:
            print("❌ Error: No data was extracted.")

    except Exception as e:
        print(f"❌ Automation Error: {e}")
    finally:
        driver.quit()

if __name__ == "__main__":
    scrape_cpi_13months()

🔄 Initializing Final Scraper (13 Months / Vertical Stacking / Year-Overflow)...
🔄 [1/13] Fetching: December 2025...
🔄 [2/13] Fetching: November 2025...
🔄 [3/13] Fetching: October 2025...
🔄 [4/13] Fetching: September 2025...
🔄 [5/13] Fetching: August 2025...
🔄 [6/13] Fetching: July 2025...
🔄 [7/13] Fetching: June 2025...
🔄 [8/13] Fetching: May 2025...
🔄 [9/13] Fetching: April 2025...
🔄 [10/13] Fetching: March 2025...
🔄 [11/13] Fetching: February 2025...
🔄 [12/13] Fetching: January 2025...
🔄 [13/13] Fetching: December 2024...

✨ Done! 13-month report saved as CPI_last13months.csv


In [50]:
import pandas as pd

def process_cpi_pipeline(input_file, output_file):
    # 1. Load the vertically stacked CSV file
    df = pd.read_csv(input_file)
    
    # 2. Pivot the table
    pivoted_df = df.pivot_table(
        index=['Year', 'Month'], 
        columns='Category', 
        values='Value'
    ).reset_index()
    
    # 3. Rename columns - SWAPPED to match user nomenclature
    pivoted_df = pivoted_df.rename(columns={
        'Non-prescribed medicines': 'CPI_Non-prescribed',
        'Prescribed medicines (excluding medicinal cannabis)5': 'CPI_Prescribed'
    })
    
    # 4. Clean up headers
    pivoted_df.columns.name = None
    
    # 5. Set chronological order for precise calculation
    months_order = [
        "January", "February", "March", "April", "May", "June", 
        "July", "August", "September", "October", "November", "December"
    ]
    pivoted_df['Month'] = pd.Categorical(pivoted_df['Month'], categories=months_order, ordered=True)
    
    # Sort Past to Present for pct_change()
    pivoted_df = pivoted_df.sort_values(by=['Year', 'Month'], ascending=[True, True])
    
    # 6. Calculate MoM Change: (Current - Previous) / Previous
    pivoted_df['Prescribed_MoM_Change'] = pivoted_df['CPI_Prescribed'].pct_change()
    pivoted_df['Non-prescribed_MoM_Change'] = pivoted_df['CPI_Non-prescribed'].pct_change()
    
    # 7. Reorder columns to put Value and Change side-by-side
    cols_order = [
        'Year', 'Month', 
        'CPI_Prescribed', 'Prescribed_MoM_Change', 
        'CPI_Non-prescribed', 'Non-prescribed_MoM_Change'
    ]
    pivoted_df = pivoted_df[cols_order]
    
    # 8. Sort Present to Past (Descending) for final view
    pivoted_df = pivoted_df.sort_values(by=['Year', 'Month'], ascending=[False, False])
    
    # 9. Save the final report
    pivoted_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"✨ Done! Processed CPI data is saved as Processed_CPI.csv")
    
    return pivoted_df

# Execute
final_report = process_cpi_pipeline("CPI_last13months.csv", "Processed_CPI.csv")
final_report

✨ Done! Processed CPI data is saved as Processed_CPI.csv


,Year,Month,CPI_Prescribed,Prescribed_MoM_Change,CPI_Non-prescribed,Non-prescribed_MoM_Change
3,2025,December,91.9,0.000000,147.0,-0.003390
10,2025,November,91.9,0.000000,147.5,-0.006734
11,2025,October,91.9,0.000000,148.5,0.002701
12,2025,September,91.9,0.000000,148.1,-0.006707
2,2025,August,91.9,0.000000,149.1,-0.008644
6,2025,July,91.9,0.000000,150.4,-0.003313
7,2025,June,91.9,0.001089,150.9,0.000000
9,2025,May,91.8,0.007684,150.9,0.027229
1,2025,April,91.1,0.002200,146.9,-0.004743
8,2025,March,90.9,0.000000,147.6,0.008197
